In [0]:
from pyspark.sql.functions import col, when
#creating the attendance band
silver_df = silver_df.withColumn(
    "attendance_band",
    when(col("attendance_percentage") >= 90, "Excellent")
    .when(col("attendance_percentage") >= 75, "Good")
    .otherwise("Poor")
)
#performance band
silver_df = silver_df.withColumn(
    "performance_band",
    when(col("overall_score") >= 85, "High")
    .when(col("overall_score") >= 60, "Medium")
    .otherwise("Low")
)
#risk status
silver_df = silver_df.withColumn(
    "risk_status",
    when(
        (col("attendance_percentage") < 75) |
        (col("overall_score") < 50),
        "At Risk"
    ).otherwise("Normal")
)

display(
    silver_df.select(
        "student_id",
        "attendance_percentage",
        "attendance_band",
        "overall_score",
        "performance_band",
        "risk_status"
    ).limit(20)
)
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "student_success.silver.student_performance"
    )

In [0]:
#removing the duplicates
silver_df = bronze_df.dropDuplicates()
print("Rows after deduplication:", silver_df.count())

In [0]:
from pyspark.sql.functions import *
#load Bronze table
bronze_df = spark.table(
    "student_success.bronze.student_performance"
)

display(bronze_df.limit(5))
# Row count
print("Rows:", bronze_df.count())

# Duplicate count
print("Distinct Rows:", bronze_df.distinct().count())

bronze_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bronze_df.columns
]).show()